# Autonomous Deal-Hunter Agent

A **multi-agent pipeline** that autonomously:
1. **Scans** RSS deal feeds (ScannerAgent)
2. **Prices** each product using an LLM (PricerAgent)
3. **Identifies** deals where estimated value > listed price by ≥ 30 % (OpportunityAgent)
4. **Notifies** you via Pushover push notification (MessengerAgent)
5. **Plans** the whole pipeline with tool-use orchestration (PlannerAgent)
6. **Displays** everything in a Gradio UI

> Pushover notifications are optional(set `PUSHOVER_TOKEN` and `PUSHOVER_USER`).

In [ ]:
import os
import json
import time
import re
import requests
from dataclasses import dataclass, field
from typing import Optional
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field
from bs4 import BeautifulSoup
import feedparser
import gradio as gr
from tqdm import tqdm

load_dotenv(override=True)
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
print(f"OpenRouter API Key: {openrouter_api_key[:4]}...")

openai_client = OpenAI(
    api_key=openrouter_api_key,
    base_url="https://openrouter.ai/api/v1"
)

PUSHOVER_TOKEN = os.getenv("PUSHOVER_TOKEN", "")
PUSHOVER_USER  = os.getenv("PUSHOVER_USER",  "")

print("Pushover configured:", bool(PUSHOVER_TOKEN and PUSHOVER_USER))

## Data Models

In [ ]:
class Deal(BaseModel):
    product_description: str = Field(
        description="3-4 sentence summary of the product. Focus on the item, not the deal."
    )
    price: float = Field(description="The listed price of this product")
    url: str     = Field(description="The URL of the deal")
    title: str   = Field(description="Short product title")


class DealSelection(BaseModel):
    deals: list[Deal] = Field(
        description="The 5 best deals with clear prices and detailed descriptions"
    )


@dataclass
class Opportunity:
    deal:      Deal
    estimate:  float
    discount:  float   # fraction, e.g. 0.35 = 35 %

    @property
    def savings(self) -> float:
        return self.estimate - self.deal.price

    def __repr__(self):
        return (
            f"<Opportunity: {self.deal.title[:50]}  "
            f"listed ${self.deal.price:.0f}  est ${self.estimate:.0f}  "
            f"({self.discount*100:.0f}% off)"
        )

## ScannerAgent

Reads RSS feeds and uses GPT with **structured output** to select the 5 best deals.

In [ ]:
FEEDS = [
    "https://www.dealnews.com/c142/Electronics/?rss=1",
    "https://www.dealnews.com/c39/Computers/?rss=1",
    "https://www.dealnews.com/f1912/Smart-Home/?rss=1",
]


def _clean_html(html: str) -> str:
    soup = BeautifulSoup(html, "html.parser")
    text = soup.get_text(separator=" ")
    return re.sub(r"\s+", " ", text).strip()[:800]


def _fetch_raw_deals(n_per_feed: int = 8) -> list[dict]:
    """Pull entries from all RSS feeds and return raw dicts."""
    raw = []
    for url in FEEDS:
        feed = feedparser.parse(url)
        for entry in feed.entries[:n_per_feed]:
            raw.append({
                "title":   entry.get("title", ""),
                "summary": _clean_html(entry.get("summary", "")),
                "url":     entry.get("link", ""),
            })
    return raw


class ScannerAgent:
    """Scans RSS feeds and selects the 5 best deals via GPT structured output."""

    SYSTEM = (
        "You are a deal-curation assistant. From the list of raw deals provided, "
        "choose the 5 most interesting, with clear prices and detailed descriptions. "
        "Focus on the product itself, not the discount. "
        "Respond with valid JSON matching the DealSelection schema: "
        '{"deals": [{"product_description": "...", "price": 0.0, "url": "...", "title": "..."}]}'
    )

    def scan(self) -> list[Deal]:
        raw_deals = _fetch_raw_deals()
        if not raw_deals:
            print("No deals fetched from feeds (check connectivity)")
            return []

        deals_text = "\n\n".join(
            f"Title: {d['title']}\nSummary: {d['summary']}\nURL: {d['url']}"
            for d in raw_deals
        )

        # Use JSON mode (works with OpenRouter) instead of the beta structured-output endpoint
        response = openai_client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": self.SYSTEM},
                {"role": "user",   "content": deals_text},
            ],
            response_format={"type": "json_object"},
        )

        try:
            data = json.loads(response.choices[0].message.content)
            return DealSelection(**data).deals
        except Exception as exc:
            print(f"[ScannerAgent] Failed to parse deals response: {exc}")
            return []


# Quick test
scanner = ScannerAgent()
print("ScannerAgent instantiated. Call scanner.scan() to fetch live deals.")

## PricerAgent

Estimates the 'fair' market value of a product using GPT.

In [ ]:
class PricerAgent:
    """Estimates the true market price of a product using a frontier LLM."""

    SYSTEM = (
        "You are an expert product appraiser. "
        "Reply with ONLY the estimated retail price in USD as a whole number — no $ sign, no explanation."
    )

    def estimate(self, deal: Deal) -> float:
        prompt = (
            f"Product: {deal.title}\n"
            f"Description: {deal.product_description}\n\n"
            "What is the typical retail price (USD) for this product?"
        )
        response = openai_client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": self.SYSTEM},
                {"role": "user",   "content": prompt},
            ],
            temperature=0,
        )
        try:
            return float(response.choices[0].message.content.strip().replace(",", ""))
        except ValueError:
            return deal.price  # fallback: no discount


pricer = PricerAgent()
print("PricerAgent instantiated.")

## MessengerAgent

Sends push notifications via the Pushover API.

In [ ]:
class MessengerAgent:
    """Sends push notifications for great opportunities via Pushover."""

    PUSHOVER_URL = "https://api.pushover.net/1/messages.json"
    DISCOUNT_THRESHOLD = 0.30   # 30 % minimum discount

    def notify(self, opportunity: Opportunity) -> bool:
        if not (PUSHOVER_TOKEN and PUSHOVER_USER):
            print(f"[Messenger] Pushover not configured — skipping notification for: {opportunity.deal.title[:50]}")
            return False

        if opportunity.discount < self.DISCOUNT_THRESHOLD:
            return False  # not a good enough deal

        message = (
            f"🛍️ DEAL ALERT: {opportunity.deal.title[:80]}\n"
            f"Listed: ${opportunity.deal.price:.0f}  "
            f"Estimated: ${opportunity.estimate:.0f}  "
            f"Savings: ${opportunity.savings:.0f} ({opportunity.discount*100:.0f}% off)\n"
            f"{opportunity.deal.url}"
        )
        resp = requests.post(self.PUSHOVER_URL, data={
            "token":   PUSHOVER_TOKEN,
            "user":    PUSHOVER_USER,
            "message": message,
            "title":   "Price is Right — Deal Found!",
        })
        return resp.status_code == 200


messenger = MessengerAgent()
print("MessengerAgent instantiated.")

## PlannerAgent (Tool-Use Orchestration)

Uses GPT function-calling to chain the scan → price → notify pipeline autonomously.

In [ ]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "scan_deals",
            "description": "Scan RSS deal feeds and return the top 5 curated deals.",
            "parameters": {"type": "object", "properties": {}, "required": []},
        },
    },
    {
        "type": "function",
        "function": {
            "name": "estimate_price",
            "description": "Estimate the fair market price for a deal.",
            "parameters": {
                "type": "object",
                "properties": {
                    "deal_index": {"type": "integer", "description": "Index (0-4) of the deal to price"},
                },
                "required": ["deal_index"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "send_notification",
            "description": "Send a push notification for a great deal opportunity.",
            "parameters": {
                "type": "object",
                "properties": {
                    "deal_index": {"type": "integer", "description": "Index of the deal"},
                },
                "required": ["deal_index"],
            },
        },
    },
]

import openai as _openai

def _call_with_retry(max_retries: int = 3, wait: float = 2.0, **kwargs):
    """Call openai_client.chat.completions.create with exponential-backoff retry on 5xx errors."""
    last_exc = None
    for attempt in range(max_retries):
        try:
            return openai_client.chat.completions.create(**kwargs)
        except (_openai.InternalServerError, _openai.APIStatusError) as exc:
            if exc.status_code and exc.status_code < 500:
                raise           # 4xx errors are not retryable
            last_exc = exc
            if attempt < max_retries - 1:
                time.sleep(wait * (2 ** attempt))
    raise last_exc


class PlannerAgent:
    """Orchestrates the full deal-hunting pipeline via tool-use."""

    SYSTEM = (
        "You are an autonomous deal-hunting agent. Your goal:\n"
        "1. Call scan_deals to get the current top deals.\n"
        "2. Call estimate_price for EACH deal to determine its fair value.\n"
        "3. For any deal where estimated value > listed price by 30 %+, call send_notification.\n"
        "4. Report a summary of what you found and which deals were worth notifying about."
    )

    def __init__(self):
        self._scanner   = ScannerAgent()
        self._pricer    = PricerAgent()
        self._messenger = MessengerAgent()
        self._deals: list[Deal] = []
        self._opportunities: list[Opportunity] = []

    def _handle_tool(self, name: str, args: dict) -> str:
        if name == "scan_deals":
            self._deals = self._scanner.scan()
            if not self._deals:
                return "No deals found (check feed connectivity)."
            return json.dumps([{"index": i, "title": d.title, "price": d.price} for i, d in enumerate(self._deals)])

        elif name == "estimate_price":
            idx  = args["deal_index"]
            deal = self._deals[idx]
            est  = self._pricer.estimate(deal)
            discount = (est - deal.price) / est if est > 0 else 0
            opp = Opportunity(deal=deal, estimate=est, discount=max(discount, 0))
            self._opportunities.append(opp)
            return json.dumps({"index": idx, "estimate": est, "discount_pct": round(discount * 100, 1)})

        elif name == "send_notification":
            idx = args["deal_index"]
            matching = [o for o in self._opportunities if o.deal.url == self._deals[idx].url]
            if matching:
                sent = self._messenger.notify(matching[0])
                return json.dumps({"sent": sent})
            return json.dumps({"sent": False, "reason": "opportunity not found"})

        return json.dumps({"error": f"Unknown tool: {name}"})

    def run(self, log_fn=print) -> list[Opportunity]:
        """Execute the autonomous planning loop."""
        messages = [
            {"role": "system", "content": self.SYSTEM},
            {"role": "user",   "content": "Start the deal-hunting pipeline now."},
        ]

        for step in range(20):  # safety limit
            response = _call_with_retry(
                model="gpt-4o-mini",
                messages=messages,
                tools=TOOLS,
                tool_choice="auto",
            )
            msg = response.choices[0].message
            messages.append(msg)

            if not msg.tool_calls:
                log_fn(f"\n🤖 Planner summary:\n{msg.content}")
                break

            for tc in msg.tool_calls:
                args   = json.loads(tc.function.arguments)
                result = self._handle_tool(tc.function.name, args)
                log_fn(f"  → {tc.function.name}({args}) = {result[:120]}")
                messages.append({
                    "role":         "tool",
                    "tool_call_id": tc.id,
                    "content":      result,
                })

        return self._opportunities


print("PlannerAgent defined.")

## Quick Smoke-Test (runs the full pipeline once)

In [ ]:
planner = PlannerAgent()
opportunities = planner.run()

print(f"\n✅ Found {len(opportunities)} priced opportunities")
great_deals = [o for o in opportunities if o.discount >= 0.30]
print(f"   {len(great_deals)} qualify as great deals (≥30% off)")

for opp in sorted(opportunities, key=lambda o: o.discount, reverse=True):
    print(f"  {'🔥' if opp.discount >= 0.30 else '  '} {opp.deal.title[:60]:<60}  "
          f"listed ${opp.deal.price:>6.0f}  est ${opp.estimate:>6.0f}  {opp.discount*100:>5.1f}% off")

## Gradio UI — Browse & Hunt Deals

In [ ]:
import pandas as pd

CSS = """
.gradio-container { max-width: 100% !important; }
.hunt-btn button { background: #753991 !important; color: white !important; font-weight: 700; }
"""


def run_hunt(progress=gr.Progress()):
    """Execute the full agent pipeline and return results as a dataframe."""
    logs = []
    progress(0, desc="Starting deal hunter...")

    agent = PlannerAgent()
    opps  = agent.run(log_fn=lambda m: logs.append(m))

    progress(0.9, desc="Building results table...")

    if not opps:
        df = pd.DataFrame(columns=["Deal", "Title", "Listed $", "Estimated $", "Savings $", "Discount %", "URL"])
    else:
        rows = []
        for o in sorted(opps, key=lambda x: x.discount, reverse=True):
            rows.append({
                "Deal":        "🔥" if o.discount >= 0.30 else "",
                "Title":       o.deal.title[:80],
                "Listed $":    f"${o.deal.price:.0f}",
                "Estimated $": f"${o.estimate:.0f}",
                "Savings $":   f"${o.savings:.0f}",
                "Discount %":  f"{o.discount*100:.1f}%",
                "URL":         o.deal.url,
            })
        df = pd.DataFrame(rows)

    progress(1.0, desc="Done!")
    log_text = "\n".join(logs)
    return df, log_text


with gr.Blocks(css=CSS, theme=gr.themes.Monochrome(), title="Price is Right — Deal Hunter") as ui:
    gr.Markdown("# 🛍️ Autonomous Deal Hunter — Week 8")
    gr.Markdown(
        "Click **Hunt Deals** to run the full agent pipeline: scan RSS feeds → "
        "price each item → identify bargains → send notifications."
    )

    with gr.Row():
        hunt_btn = gr.Button("🔍 Hunt Deals", elem_classes=["hunt-btn"])

    deals_table = gr.Dataframe(
        label="Opportunities (sorted by discount %)",
        wrap=True,
    )

    agent_log = gr.Textbox(
        label="Agent Activity Log",
        lines=15,
        placeholder="Agent logs will appear here after the hunt...",
    )

    hunt_btn.click(fn=run_hunt, inputs=[], outputs=[deals_table, agent_log])

ui.launch(inbrowser=True)